In [ ]:
from sickle import Sickle
import pandas as pd
import time

sickle = Sickle('https://tailieuso.tlu.edu.vn/oai/request',verify=False)


records = sickle.ListRecords(metadataPrefix='oai_dc', ignore_deleted=True)

data = []
count = 0
max_records = 5000  

print("Đang thu thập dữ liệu...")

try:
    for record in records:
        # Lấy thông tin trong thẻ <metadata>
        if 'metadata' in record.raw:
            meta = record.metadata
            
            # Trích xuất các trường quan trọng
            title = meta.get('title', [''])[0]
            creators = '; '.join(meta.get('creator', []))
            subjects = '; '.join(meta.get('subject', [])) # Từ khóa
            description = '; '.join(meta.get('description', [])) # Tóm tắt/Abstract
            date = meta.get('date', [''])[0]
            publisher = meta.get('publisher', [''])[0]
            language = meta.get('language', [''])[0]
            identifier = meta.get('identifier', [''])[0] # Link gốc
            doc_type_list = meta.get('type', []) 
            doc_type = '; '.join(doc_type_list)
            data.append({
                'id': record.header.identifier,
                'title': title,
                'creators': creators,
                'subjects': subjects,
                'abstract': description, 
                'date': date,
                'document_type': doc_type,
                'publisher': publisher,
                'language':     language,
                'link': identifier
            })
            
            count += 1
            if count % 100 == 0:
                print(f"Đã lấy {count} bản ghi...")
            
            if count >= max_records:
                break
                
except Exception as e:
    print(f"Lỗi hoặc hết dữ liệu: {e}")


df = pd.DataFrame(data)
df.to_csv('tlu_metadata.csv', index=False, encoding='utf-8-sig')
print(f"Hoàn thành! Đã lưu {len(df)} dòng vào file tlu_metadata.csv")


In [2]:
from sickle import Sickle

sickle = Sickle('https://tailieuso.tlu.edu.vn/oai/request', verify=False)

print("=== CÁC METADATA FORMAT HỖ TRỢ ===\n")

formats = sickle.ListMetadataFormats()

for fmt in formats:
    print(f"Prefix: {fmt.metadataPrefix}")
    print(f"  Schema: {fmt.schema}")
    print(f"  Namespace: {fmt.metadataNamespace}")
    print("-" * 70)

=== CÁC METADATA FORMAT HỖ TRỢ ===



c:\Program Files\Python312\Lib\site-packages\urllib3\connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'tailieuso.tlu.edu.vn'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Prefix: uketd_dc
  Schema: http://naca.central.cranfield.ac.uk/ethos-oai/2.0/uketd_dc.xsd
  Namespace: http://naca.central.cranfield.ac.uk/ethos-oai/2.0/
----------------------------------------------------------------------
Prefix: qdc
  Schema: http://dublincore.org/schemas/xmls/qdc/2006/01/06/dcterms.xsd
  Namespace: http://purl.org/dc/terms/
----------------------------------------------------------------------
Prefix: didl
  Schema: http://standards.iso.org/ittf/PubliclyAvailableStandards/MPEG-21_schema_files/did/didl.xsd
            
  Namespace: urn:mpeg:mpeg21:2002:02-DIDL-NS
----------------------------------------------------------------------
Prefix: mods
  Schema: http://www.loc.gov/standards/mods/v3/mods-3-1.xsd
  Namespace: http://www.loc.gov/mods/v3
----------------------------------------------------------------------
Prefix: ore
  Schema: http://tweety.lanl.gov/public/schemas/2008-06/atom-tron.sch
  Namespace: http://www.w3.org/2005/Atom
-------------------------------

In [ ]:
import pandas as pd
df = pd.read_csv('tlu_metadata.csv')
df.head()


In [ ]:
df.columns

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df.document_type.value_counts()

In [28]:
df[df.document_type == 'BB; LT']

,title,creators,subjects,abstract,date,document_type,link
4281,Local Trichoderma strains as a control strateg...,"Abd-El-Kareem, Farid",Biocontrol; Black root rot; Enzyme activity; S...,Each of the antagonistic fungal strains signif...,2021-08-17T08:17:25Z,BB; LT,2522-8307


In [ ]:
import pandas as pd
df = pd.read_json('tlu_metadata_clean.jsonl', lines=True)

print(f"Tổng số document: {len(df)}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated(subset=['id']).sum()}")
print(f"\nSample authors format:\n{df['authors'].head(3)}")


Tổng số document: 5000

Null values:
id              0
uri             0
oai_id          0
type            0
raw_type        0
title           0
abstract        0
language        0
publisher       0
authors         0
advisors        0
subjects        0
year            0
date            0
degree       3247
major           0
ddc             0
dtype: int64

Duplicates: 0

Sample authors format:
0    [Đinh Văn Thạch]
1    [Lưu Mạnh Quảng]
2       [Mẫn Văn Thể]
Name: authors, dtype: object


In [1]:
import json

with open(r"D:\Document\RAG_Learning\data\tlu_metadata_clean_v2.jsonl", "r", encoding="utf-8") as f:
    data = json.loads(f.readline())

print(data.get('uri'))


http://tailieuso.tlu.edu.vn/handle/DHTL/6051


In [1]:
import asyncio
import os
from dotenv import load_dotenv
from lightrag.kg.neo4j_impl import Neo4JStorage
from lightrag.utils import EmbeddingFunc
import numpy as np

load_dotenv("D:/Document/RAG_Learning/.env")

async def dummy_embed(texts):
    return np.zeros((len(texts), 384))

embedding_func = EmbeddingFunc(
    embedding_dim=384,
    max_token_size=512,
    func=dummy_embed
)

global_config = {
    "working_dir": "./tlu_rag",
    "max_graph_nodes": 1000,
}

async def main():
    graph = Neo4JStorage(
        namespace="chunk_entity_relation",
        global_config=global_config,
        embedding_func=embedding_func,
        workspace="base"
    )
    await graph.initialize()

    print("\n=== Test has_node ===")
    # Thay bằng tên person thật trong Neo4j của bạn
    result = await graph.has_node("Đinh Văn Thạch")
    print(f"has_node('Đinh Văn Thạch'): {result}")

    print("\n=== Test get_node ===")
    node = await graph.get_node("Đinh Văn Thạch")
    print(f"get_node: {node}")

    print("\n=== Test get_node_edges ===")
    edges = await graph.get_node_edges("Đinh Văn Thạch")
    print(f"get_node_edges: {edges[:3] if edges else None}")

    print("\n=== Test has_edge ===")
    if edges:
        src, tgt = edges[0]
        result = await graph.has_edge(src, tgt)
        print(f"has_edge({src}, {tgt}): {result}")

    print("\n=== Test get_all_labels ===")
    labels = await graph.get_all_labels()
    print(f"Total labels: {len(labels)}")
    print(f"Sample: {labels[:5]}")

    await graph.finalize()

asyncio.run(main())

ModuleNotFoundError: No module named 'lightrag'